# P1F: is trader composition irrelevant for forecasting prices?

Registered test (`ceiling_suite/p1_empirics/P1F_forecast_irrelevance_spec.md`). **Run it once, top to bottom.**

1. Clones the public repository and checks that the specification and executor match the registered hashes and that the test has not been opened.
2. Downloads Binance USD-M archive files for 1 March to 29 April 2022 (checksum-verified), builds 30-minute bars, runs the power gate on the training half, then A1, A2 and the secondary results.
3. Zips every output and downloads `p1f_results.zip`.

No API keys are needed. It takes about 20 to 40 minutes. If it stops part-way, do **not** rerun: send the printed output instead.

In [ ]:
!git clone --depth 1 https://github.com/Pratikn03/Fianince.git /content/Fianince 2>&1 | tail -2
%cd /content/Fianince/ceiling_suite/p1_empirics
!git -C /content/Fianince log -1 --format='commit %H %cd'

In [ ]:
import hashlib, json
spec = json.load(open('P1F_spec.json'))
ok = True
for f, h in spec['sha256'].items():
    got = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    print(('OK  ' if got == h else 'BAD ') + f, got[:16])
    ok &= got == h
unlocks = json.load(open('P1F_unlocks.json'))
print('opens =', unlocks['opens'])
assert ok, 'A registered file has changed: stop and report.'
assert unlocks['opens'] == 0, 'P1F has already been opened: stop.'

In [ ]:
!pip -q install requests pandas numpy
import sys; sys.path.insert(0, '.')
import p1f_executor as ex
res = ex.run_all('/content/p1f_out')
print(json.dumps({k: v for k, v in res.items() if k not in ('A2_by_class',)}, indent=1)[:6000])

In [ ]:
import shutil, json
open('/content/p1f_out/p1f_printed_summary.json', 'w').write(json.dumps(res, indent=1))
shutil.make_archive('/content/p1f_results', 'zip', '/content/p1f_out')
from google.colab import files
files.download('/content/p1f_results.zip')